In [1]:
pip install sqlalchemy pymysql mysql-connector-python


Note: you may need to restart the kernel to use updated packages.


In [2]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus
import pandas as pd
from IPython.display import display

user = "khirod"
pwd = "Khirod@14324"   
host = "localhost"
port = 3306
db = "telecom_analysis"

uri = f"mysql+pymysql://{user}:{quote_plus(pwd)}@{host}:{port}/{db}"

engine = create_engine(uri)

print("Connected successfully!")



Connected successfully!


In [3]:
df = pd.read_csv("../DATA/CLEANED/telecom_feature_engineered_cleaned.csv")


In [4]:
df.to_sql(
    name="telecom_analysis",
    con=engine,
    if_exists="replace",
    index=False
)


7043

In [25]:
pd.read_sql("""
SELECT 
    COUNT(*) AS total_customers,
    ROUND(AVG(churn_flag) * 100, 2) AS churn_rate_percent,
    ROUND(SUM(revenue_at_risk), 2) AS total_revenue_at_risk
FROM telecom_analysis;
""", engine)


,total_customers,churn_rate_percent,total_revenue_at_risk
0,7043,26.54,139130.85


**Insights :**

The overall customer base consists of 7,043 customers, with an overall churn rate of 26.54%, indicating a moderate level of customer attrition.

Customer churn translates into an estimated total revenue at risk of 139,130.85, underscoring the significant financial impact of retention challenges.

These baseline metrics provide a clear benchmark against which high-risk segments and targeted retention strategies can be evaluated.

In [13]:
pd.read_sql("""
SELECT
    tenure_bucket,
    COUNT(*) AS customer_count,
    ROUND(AVG(churn_flag) * 100, 2) AS churn_rate_percent,
    ROUND(SUM(revenue_at_risk), 2) AS revenue_at_risk
FROM telecom_analysis
GROUP BY tenure_bucket
ORDER BY revenue_at_risk DESC;
""", engine)


,tenure_bucket,customer_count,churn_rate_percent,revenue_at_risk
0,0-6 months,1481,52.94,49896.10
1,24+ months,3833,14.04,47094.95
2,12-24 months,1024,28.71,23081.65
3,6-12 months,705,35.89,19058.15


**Insights :**

Customers in the 0–6 months tenure cohort exhibit an exceptionally high churn rate (~53%), confirming that the early lifecycle stage is the most vulnerable period for attrition.

Despite having a smaller customer base, the 0–6 months cohort contributes the highest revenue at risk, making early-stage churn both a behavioral and financial concern.

Long-tenure customers (24+ months) show significantly lower churn (~14%), indicating that retention stabilizes substantially once customers pass the initial engagement phase.

In [17]:
pd.read_sql("""
SELECT
    contract_risk,
    COUNT(*) AS customer_count,
    ROUND(AVG(churn_flag) * 100, 2) AS churn_rate_percent,
    ROUND(SUM(revenue_at_risk), 2) AS revenue_at_risk
FROM telecom_analysis
GROUP BY contract_risk
ORDER BY churn_rate_percent DESC;
""", engine)


,contract_risk,customer_count,churn_rate_percent,revenue_at_risk
0,High,3875,42.71,120847.10
1,Medium,1473,11.27,14118.45
2,Low,1695,2.83,4165.30


**Insights :**

Customers on high-risk (month-to-month) contracts exhibit a substantially higher churn rate (~43%) compared to those on longer-term contracts, highlighting contract type as a primary churn driver.

Although high-risk contracts represent just over half of the customer base, they account for the vast majority of revenue at risk, indicating a strong concentration of financial loss within this segment.

Low-risk (long-term) contracts show minimal churn (~3%), reinforcing the effectiveness of longer contract commitments in stabilizing customer retention.

In [30]:
pd.read_sql("""
SELECT
    price_sensitivity_flag,
    COUNT(*) AS customer_count,
    ROUND(AVG(churn_flag) * 100, 2) AS churn_rate_percent,
    ROUND(SUM(revenue_at_risk), 2) AS revenue_at_risk
FROM telecom_analysis
GROUP BY price_sensitivity_flag;
""", engine)


,price_sensitivity_flag,customer_count,churn_rate_percent,revenue_at_risk
0,0,6838,25.10,124352.25
1,1,205,74.63,14778.60


**Insights:**

Price-sensitive customers (flag = 1) exhibit an extremely high churn rate (~74.6%), indicating strong price-driven attrition despite representing a small portion of the customer base.

Although price-sensitive customers account for only ~3% of customers, they contribute disproportionately to churn risk, making them a high-impact segment for targeted pricing or discount interventions.

Non–price-sensitive customers show significantly lower churn (~25%), suggesting that pricing stability plays a critical role in long-term retention.

In [39]:
pd.read_sql("""
WITH clv_ranked AS (
    SELECT
        customer_lifetime_value,
        churn_flag,
        revenue_at_risk,
        NTILE(4) OVER (ORDER BY customer_lifetime_value) AS quartile
    FROM telecom_analysis
)
SELECT
    CASE 
        WHEN quartile = 4 THEN 'High Value'
        ELSE 'Low Value'
    END AS customer_segment,
    COUNT(*) AS customer_count,
    ROUND(AVG(churn_flag) * 100, 2) AS churn_rate_percent,
    ROUND(SUM(revenue_at_risk), 2) AS revenue_at_risk
FROM clv_ranked
GROUP BY customer_segment;

""", engine)


,customer_segment,customer_count,churn_rate_percent,revenue_at_risk
0,Low Value,5283,30.55,113790.65
1,High Value,1760,14.49,25340.20


**Insight:**

High-value customers exhibit a significantly lower churn rate (~14.5%) compared to low-value customers, indicating stronger loyalty among customers with higher lifetime value.

Despite lower churn rates, high-value customers still contribute meaningful revenue at risk, making their retention strategically important due to their higher revenue impact per customer.

Low-value customers account for the majority of churn-related revenue loss, suggesting that broad churn reduction efforts should prioritize improving engagement within this larger segment while selectively protecting high-value users.


In [42]:
pd.read_sql("""
SELECT
    tenure_bucket,
    ROUND(SUM(revenue_at_risk) * 100.0 / 
          (SELECT SUM(revenue_at_risk) FROM telecom_analysis), 2)
          AS revenue_risk_share_pct
FROM telecom_analysis
GROUP BY tenure_bucket
ORDER BY revenue_risk_share_pct DESC;
""", engine)


,tenure_bucket,revenue_risk_share_pct
0,0-6 months,35.86
1,24+ months,33.85
2,12-24 months,16.59
3,6-12 months,13.70


**Insight:**

The 0–6 months tenure cohort contributes the largest share of revenue at risk (~36%), confirming that early-stage churn is the most financially damaging.

Although 24+ months customers have much lower churn rates, they still account for a significant portion of revenue at risk (~34%) due to their larger customer base and higher accumulated value.

Together, early-tenure and long-tenure customers drive nearly 70% of total revenue risk, indicating that retention strategies should balance early onboarding improvements with long-term customer engagement.

In [47]:
pd.read_sql("""
SELECT
    tenure_bucket,
    contract_risk,
    COUNT(*) AS high_risk_customers,
    ROUND(AVG(churn_flag) * 100, 2) AS churn_rate_percent
FROM telecom_analysis
WHERE churn_flag = 1
GROUP BY tenure_bucket, contract_risk
ORDER BY high_risk_customers DESC;
""", engine)


,tenure_bucket,contract_risk,high_risk_customers,churn_rate_percent
0,0-6 months,High,780,100.0
1,24+ months,High,353,100.0
2,12-24 months,High,278,100.0
3,6-12 months,High,244,100.0
4,24+ months,Medium,137,100.0
5,24+ months,Low,48,100.0
6,12-24 months,Medium,16,100.0
7,6-12 months,Medium,9,100.0
8,0-6 months,Medium,4,100.0


**Insights:**

The majority of churned customers belong to high-risk contracts across all tenure groups, highlighting contract type as the dominant driver of churn.

Early-tenure customers (0–6 months) on high-risk contracts represent the largest churn volume, making this segment the highest-priority target for retention interventions.

Churn also persists among long-tenure customers on high-risk contracts, indicating that contract structure can override customer loyalty over time.

In [58]:
pd.read_sql("""
SELECT
    '0-6 months' AS tenure_segment,
    COUNT(*) AS customer_count,
    ROUND(AVG(churn_flag) * 100, 2) AS churn_rate_percent,
    ROUND(SUM(revenue_at_risk), 2) AS revenue_at_risk
FROM telecom_analysis
WHERE tenure_bucket = '0-6 months'

UNION ALL

SELECT
    '24+ months' AS tenure_segment,
    COUNT(*) AS customer_count,
    ROUND(AVG(churn_flag) * 100, 2) AS churn_rate_percent,
    ROUND(SUM(revenue_at_risk), 2) AS revenue_at_risk
FROM telecom_analysis
WHERE tenure_bucket = '24+ months';
""", engine)


,tenure_segment,customer_count,churn_rate_percent,revenue_at_risk
0,0-6 months,1481,52.94,49896.10
1,24+ months,3833,14.04,47094.95


**Insight:**

Early-stage customers (0–6 months) experience extremely high churn and contribute the highest revenue loss, making onboarding the most critical retention phase.

Long-tenure customers (24+ months) show significantly lower churn, confirming that customer stability increases with tenure.

The contrast highlights that preventing early churn delivers the greatest immediate retention ROI.

In [55]:
pd.read_sql("""
SELECT
    payment_method,
    COUNT(*) AS customer_count,
    ROUND(AVG(churn_flag) * 100, 2) AS churn_rate_percent,
    ROUND(SUM(revenue_at_risk), 2) AS revenue_at_risk
FROM telecom_analysis
GROUP BY payment_method
ORDER BY churn_rate_percent DESC;
""", engine)


,payment_method,customer_count,churn_rate_percent,revenue_at_risk
0,Electronic check,2365,45.29,84288.75
1,Mailed check,1612,19.11,16803.60
2,Bank transfer (automatic),1544,16.71,20091.90
3,Credit card (automatic),1522,15.24,17946.60


**Insight:**

Customers using electronic check exhibit a significantly higher churn rate (~45%) compared to all other payment methods, indicating strong billing-related friction.

Electronic check users also contribute the largest share of revenue at risk, making this payment method a key area for churn reduction initiatives.

Automatic payment methods (bank transfer and credit card) show substantially lower churn, suggesting that encouraging auto-pay adoption could improve customer retention.